If you want to search **both nested `JSONB` fields** and **`pgvector` embeddings** together in PostgreSQL, the usual production pattern is:

* Store metadata/features in `JSONB`
* Store embeddings in `vector`
* Use:

  * `GIN` index for JSONB
  * `HNSW` or `IVFFlat` index for vectors
* Combine:

  * structured filtering (`WHERE`)
  * semantic similarity (`ORDER BY embedding <-> query_vector`)

This is extremely common in:

* retail search
* RAG systems
* recommendation engines
* catalog search
* AI agents with metadata filtering

---

# 1. Example Table Design

```sql
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE products (
    id BIGSERIAL PRIMARY KEY,

    title TEXT,

    metadata JSONB,

    embedding VECTOR(768)
);
```

---

# 2. Example JSONB Structure

```json
{
  "brand": "Nike",
  "category": {
    "main": "Shoes",
    "sub": "Running"
  },
  "price": 4999,
  "attributes": {
    "color": "black",
    "size": 10
  },
  "tags": ["sports", "running", "men"]
}
```

---

# 3. Important Indexes

## JSONB GIN Index

```sql
CREATE INDEX idx_products_metadata
ON products
USING GIN(metadata);
```

This accelerates:

* containment
* key existence
* nested lookups

---

## pgvector HNSW Index

```sql
CREATE INDEX idx_products_embedding
ON products
USING hnsw (embedding vector_cosine_ops);
```

Or IVFFlat:

```sql
CREATE INDEX idx_products_embedding_ivf
ON products
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);
```

---

# 4. Search Nested JSONB Fields

---

## A. Exact Nested Match

### Find all running shoes

```sql
SELECT *
FROM products
WHERE metadata->'category'->>'sub' = 'Running';
```

---

## B. Numeric Filter

```sql
SELECT *
FROM products
WHERE (metadata->>'price')::INT < 6000;
```

---

## C. Nested Attribute Search

```sql
SELECT *
FROM products
WHERE metadata->'attributes'->>'color' = 'black';
```

---

## D. Array Contains

```sql
SELECT *
FROM products
WHERE metadata->'tags' ? 'sports';
```

OR

```sql
SELECT *
FROM products
WHERE metadata @> '{"tags":["sports"]}';
```

---

# 5. Combine JSONB + pgvector Search

This is the real-world AI retrieval pattern.

---

## Example:

### "Find black Nike running shoes semantically similar to query"

```sql
SELECT
    id,
    title,
    metadata,
    embedding <=> '[0.12, 0.44, ...]' AS distance
FROM products
WHERE
    metadata->>'brand' = 'Nike'
    AND metadata->'category'->>'sub' = 'Running'
    AND metadata->'attributes'->>'color' = 'black'
ORDER BY embedding <=> '[0.12, 0.44, ...]'
LIMIT 10;
```

---

# 6. Better Production Pattern (Pre-filter → Vector Search)

Vector search is expensive.

Best practice:

```sql
WITH filtered AS (
    SELECT *
    FROM products
    WHERE
        metadata->>'brand' = 'Nike'
        AND metadata->'category'->>'sub' = 'Running'
)
SELECT
    *,
    embedding <=> '[0.12, 0.44, ...]' AS score
FROM filtered
ORDER BY embedding <=> '[0.12, 0.44, ...]'
LIMIT 10;
```

Why?

* Reduces vector comparisons
* Much faster at scale
* Better recall/latency balance

---

# 7. Hybrid Search (BM25 + JSONB + Vector)

Very common in retail AI systems.

```sql
SELECT
    id,
    title,
    ts_rank(search_tsv, query) * 0.3 +
    (1 - (embedding <=> query_embedding)) * 0.7 AS final_score
FROM products,
     plainto_tsquery('running shoes') query,
     (
       SELECT '[0.12, 0.44, ...]'::vector AS query_embedding
     ) q
WHERE
    search_tsv @@ query
    AND metadata->>'brand' = 'Nike'
ORDER BY final_score DESC
LIMIT 20;
```

This gives:

* keyword relevance
* semantic relevance
* metadata filtering

Exactly what large retail/search systems do.

---

# 8. Fast Nested JSONB Search with Expression Indexes

If a nested field is queried frequently:

```sql
CREATE INDEX idx_brand
ON products ((metadata->>'brand'));
```

Nested:

```sql
CREATE INDEX idx_subcategory
ON products ((metadata->'category'->>'sub'));
```

This is MUCH faster than generic GIN for hot filters.

---

# 9. Best Production Architecture

Typical enterprise retrieval pipeline:

```text
User Query
    ↓
LLM Intent Extraction
    ↓
Structured Filters
(brand/category/price)
    ↓
JSONB Pre-filter
    ↓
Vector Similarity Search
(pgvector)
    ↓
Hybrid Ranker
    ↓
Final Results
```

This is how:

* Amazon-style retrieval
* AI retail search
* RAG with metadata
* recommendation engines
  work internally.

---

# 10. Advanced: JSONPath Queries

PostgreSQL supports JSONPath.

---

## Example

```sql
SELECT *
FROM products
WHERE metadata @? '$.attributes.color == "black"';
```

Nested:

```sql
SELECT *
FROM products
WHERE metadata @? '$.category.sub == "Running"';
```

Useful for:

* dynamic filter builders
* agentic querying
* arbitrary nested schemas

---

# 11. Real AI Retail Example

Imagine query:

> "cheap black running shoes for men"

Your system might:

---

## LLM extracts:

```json
{
  "category": "Running",
  "color": "black",
  "gender": "men",
  "max_price": 5000
}
```

---

## SQL

```sql
SELECT
    id,
    title,
    embedding <=> :query_embedding AS distance
FROM products
WHERE
    metadata->'category'->>'sub' = 'Running'
    AND metadata->'attributes'->>'color' = 'black'
    AND metadata->>'gender' = 'men'
    AND (metadata->>'price')::INT <= 5000
ORDER BY embedding <=> :query_embedding
LIMIT 20;
```

This is the exact retrieval pattern used in modern AI commerce systems.

---

# 12. Important Performance Advice

## DO NOT

```sql
ORDER BY embedding <=> query_vector
```

on the entire table blindly.

---

## ALWAYS

1. Metadata filter first
2. Reduce candidate set
3. Then vector search

Otherwise latency explodes at scale.

---

# 13. Recommended Index Strategy

| Use Case            | Index            |
| ------------------- | ---------------- |
| General JSON search | GIN(jsonb)       |
| Frequent nested key | Expression index |
| Vector ANN          | HNSW             |
| Huge datasets       | IVFFlat          |
| Text search         | GIN(tsvector)    |

---

# 14. Enterprise-Level Optimization

At large scale:

```text
JSONB Filter
    ↓
Candidate IDs
    ↓
Approx ANN Search
    ↓
Cross Encoder Re-rank
    ↓
Business Rules
```

Common stack:

* PostgreSQL
* pgvector
* Redis cache
* reranker model
* FastAPI retrieval layer

Very similar to the hybrid RL-based retail retrieval system you were discussing earlier.
